# Predicción de Consumo de Alcohol — Rusia 2024

Red neuronal profunda para predecir el consumo de alcohol puro per cápita
en regiones de Rusia usando datos históricos 2017–2023.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

# Agregar raíz del proyecto al path
sys.path.insert(0, str(Path.cwd()))

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from sklearn.linear_model import LinearRegression

from src.config import Config
from src.data.loader import CSVLoader
from src.data.preprocessor import DataPreprocessor
from src.data.splitter import DataSplitter
from src.data.dataset import AlcoholDataset
from src.model.architecture import AlcoholPredictor
from src.training.trainer import Trainer
from src.evaluate.metrics import Evaluator
from src.predict.predictor import Predictor
from src.visualize import (
    plot_training_history,
    plot_actual_vs_predicted,
    plot_ranking,
    plot_metrics_comparison,
)

%matplotlib inline
sns.set_theme(style="whitegrid")
print("Setup completo")

## 2. Carga de Datos

In [ ]:
config = Config()
config.output_dir.mkdir(parents=True, exist_ok=True)

loader = CSVLoader(config.data_dir / config.raw_data_file)
df = loader.load()

print(f"Filas: {len(df):,}")
print(f"Columnas: {list(df.columns)}")
print(f"\nTipos de bebida:\n{df['Type of alcoholic beverages'].value_counts()}")
print(f"\nRegiones únicas: {df['Region'].nunique()}")
print(f"\nAños: {sorted(df['Year'].unique())}")
df.head()

### Exploración rápida

In [ ]:
df.describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Consumo promedio por tipo de bebida
ax = axes[0]
pivot_type = df.groupby("Type of alcoholic beverages")[
    "Consumption (liters of pure alcohol per capita)"
].mean().sort_values()
pivot_type.plot(kind="barh", ax=ax, color="steelblue")
ax.set_title("Consumo promedio 2017-2023 por tipo de bebida")
ax.set_xlabel("Litros de alcohol puro per cápita")

# Consumo promedio por año
ax = axes[1]
pivot_year = df.groupby("Year")[
    "Consumption (liters of pure alcohol per capita)"
].mean()
pivot_year.plot(kind="line", marker="o", ax=ax, color="crimson")
ax.set_title("Consumo promedio por año")
ax.set_ylabel("Litros de alcohol puro per cápita")

plt.tight_layout()
plt.show()

## 3. Preprocesamiento

In [ ]:
preprocessor = DataPreprocessor()
pivoted = preprocessor.pivot_data(df)
print(f"Muestras (región × bebida): {len(pivoted)}")
pivoted.head()

In [ ]:
X, y = preprocessor.extract_train_data(pivoted)
print(f"Features: {X.shape[1]}")
print(f"Target: {y.name}")
X.head()

In [ ]:
splitter = DataSplitter(
    test_size=config.test_size,
    val_size=config.val_size,
    random_state=config.random_state,
)
splits = splitter.split(X, y)

print(f"Train: {len(splits['X_train'])}")
print(f"Val:   {len(splits['X_val'])}")
print(f"Test:  {len(splits['X_test'])}")

In [ ]:
X_train = preprocessor.fit_transform(splits["X_train"])
X_val = preprocessor.transform(splits["X_val"])
X_test = preprocessor.transform(splits["X_test"])
y_train = splits["y_train"].values
y_val = splits["y_val"].values
y_test = splits["y_test"].values

train_dataset = AlcoholDataset(X_train, y_train)
val_dataset = AlcoholDataset(X_val, y_val)
test_dataset = AlcoholDataset(X_test, y_test)

input_dim = X_train.shape[1]
print(f"Dimensión de entrada: {input_dim}")
print(f"X_train shape: {X_train.shape}")

## 4. Construcción del Modelo

In [ ]:
model = AlcoholPredictor(
    input_dim=input_dim,
    hidden_dims=config.hidden_dims,
    dropout=config.dropout_rate,
)
total_params = sum(p.numel() for p in model.parameters())
print(f"Arquitectura: {input_dim} → {config.hidden_dims} → 1")
print(f"Parámetros: {total_params:,}")
model

## 5. Entrenamiento

In [ ]:
trainer = Trainer(model, train_dataset, val_dataset, config)
history = trainer.train()

final_val_loss = min(history["val_loss"])
best_epoch = np.argmin(history["val_loss"]) + 1
print(f"Mejor validation MSE: {final_val_loss:.6f} (época {best_epoch})")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(history["train_loss"], label="Train MSE", alpha=0.8)
ax.plot(history["val_loss"], label="Val MSE", alpha=0.8)
ax.axvline(best_epoch - 1, color="gray", linestyle="--", alpha=0.5, label=f"Mejor época ({best_epoch})")
ax.set_xlabel("Época")
ax.set_ylabel("MSE")
ax.set_title("Historial de entrenamiento")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Evaluación

In [ ]:
evaluator = Evaluator()

model.load_state_dict(torch.load(config.output_dir / "best_model.pth", weights_only=True))
nn_metrics = evaluator.evaluate(model, test_dataset)
print("── Neural Network ──")
for name, val in nn_metrics.items():
    print(f"  {name}: {val:.4f}")

In [ ]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
baseline_metrics = evaluator.evaluate_sklearn(baseline, X_test, y_test)
print("── Linear Regression (baseline) ──")
for name, val in baseline_metrics.items():
    print(f"  {name}: {val:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

y_test_pred_nn = model(torch.tensor(X_test, dtype=torch.float32)).detach().numpy().flatten()
y_test_pred_base = baseline.predict(X_test)

for ax, y_pred, title in zip(
    axes,
    [y_test_pred_nn, y_test_pred_base],
    ["Neural Network", "Linear Regression"],
):
    ax.scatter(y_test, y_pred, alpha=0.6, edgecolors="k", linewidth=0.5)
    min_val = min(y_test.min(), y_pred.min())
    max_val = max(y_test.max(), y_pred.max())
    ax.plot([min_val, max_val], [min_val, max_val], "r--", alpha=0.7)
    ax.set_xlabel("Real")
    ax.set_ylabel("Predicho")
    ax.set_title(title)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
metrics_names = list(nn_metrics.keys())
x = np.arange(len(metrics_names))
width = 0.35

nn_values = [nn_metrics[m] for m in metrics_names]
base_values = [baseline_metrics[m] for m in metrics_names]

bars1 = ax.bar(x - width/2, nn_values, width, label="Neural Network", color="steelblue")
bars2 = ax.bar(x + width/2, base_values, width, label="Linear Regression", color="coral")
ax.set_xticks(x)
ax.set_xticklabels(metrics_names)
ax.set_ylabel("Valor")
ax.set_title("Comparación de métricas")
ax.legend()

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f"{height:.3f}", xy=(bar.get_x() + bar.get_width()/2, height),
                    xytext=(0, 3), textcoords="offset points", ha="center", fontsize=9)

plt.tight_layout()
plt.show()

## 7. Predicción 2024

In [ ]:
predictor = Predictor(config, preprocessor, model)
predictions = predictor.predict_2024(pivoted)
print(f"Predicciones generadas: {len(predictions)}")
predictions.head(10)

In [ ]:
region_ranking = predictor.ranking_by_region(predictions)
print("Top 10 regiones con mayor consumo estimado 2024:")
display(region_ranking.head(10))

In [ ]:
beverage_ranking = predictor.ranking_by_beverage(predictions)
print("Consumo estimado 2024 por tipo de bebida:")
display(beverage_ranking)

### Ranking visual

In [ ]:
fig, ax = plt.subplots(figsize=(10, 12))
top_n = 20
top_regions = region_ranking.head(top_n)
ax.barh(range(top_n), top_regions["Avg_Pure_Alcohol_2024"].values[::-1], color="steelblue")
ax.set_yticks(range(top_n))
ax.set_yticklabels(top_regions["Region"].values[::-1])
ax.set_xlabel("Litros de alcohol puro per cápita")
ax.set_title(f"Top {top_n} regiones — Consumo estimado 2024")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
colors = plt.cm.Set2(np.linspace(0, 1, len(beverage_ranking)))
ax.barh(beverage_ranking["Type of alcoholic beverages"],
        beverage_ranking["Avg_Pure_Alcohol_2024"],
        color=colors)
ax.set_xlabel("Litros de alcohol puro per cápita")
ax.set_title("Consumo estimado 2024 por tipo de bebida")
for i, v in enumerate(beverage_ranking["Avg_Pure_Alcohol_2024"]):
    ax.text(v + 0.01, i, f"{v:.3f}", va="center", fontsize=10)
plt.tight_layout()
plt.show()

## 8. Análisis Libre

Exploración adicional de los datos y resultados.

### 8.1 Distribución del target

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
ax.hist(y_test, bins=20, alpha=0.6, label="Real", color="steelblue", edgecolor="white")
ax.hist(y_test_pred_nn, bins=20, alpha=0.6, label="Predicho NN", color="coral", edgecolor="white")
ax.set_xlabel("Litros de alcohol puro per cápita")
ax.set_ylabel("Frecuencia")
ax.set_title("Distribución: Real vs Predicho (test)")
ax.legend()

ax = axes[1]
residuals = y_test - y_test_pred_nn
ax.scatter(y_test_pred_nn, residuals, alpha=0.6, edgecolors="k", linewidth=0.5)
ax.axhline(0, color="red", linestyle="--", alpha=0.5)
ax.set_xlabel("Predicho")
ax.set_ylabel("Residual")
ax.set_title("Gráfico de residuales")

plt.tight_layout()
plt.show()

### 8.2 Error por tipo de bebida

In [ ]:
# Mapear cada muestra de test de vuelta a su tipo de bebida
pivoted_test = pivoted.iloc[splits["test_idx"]]
beverage_types_test = pivoted_test["Type of alcoholic beverages"].values

error_by_beverage = pd.DataFrame({
    "beverage": beverage_types_test,
    "residual": residuals,
})

fig, ax = plt.subplots(figsize=(10, 4))
sns.boxplot(data=error_by_beverage, x="beverage", y="residual", ax=ax, palette="Set2")
ax.axhline(0, color="red", linestyle="--", alpha=0.4)
ax.set_title("Residuales por tipo de bebida")
ax.set_xlabel("")
plt.tight_layout()
plt.show()

### 8.3 Correlaciones entre años

In [ ]:
year_cols = [f"alc_{y}" for y in config.feature_years + [config.target_year]]
corr_data = pivoted[year_cols].dropna()
corr_matrix = corr_data.corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt=".3f", cmap="coolwarm",
            vmin=-1, vmax=1, center=0, ax=ax)
ax.set_title("Correlación entre años")
plt.tight_layout()
plt.show()

### 8.4 Evolución temporal por bebida (top regiones)

In [ ]:
top_regions_list = region_ranking.head(5)["Region"].tolist()
df_top = df[df["Region"].isin(top_regions_list)]

g = sns.FacetGrid(df_top, col="Region", hue="Type of alcoholic beverages",
                   col_wrap=3, height=3.5, sharey=False)
g.map(sns.lineplot, "Year", "Consumption (liters of pure alcohol per capita)", marker="o")
g.add_legend()
g.set_titles("{col_name}")
g.set_axis_labels("Año", "Alcohol puro per cápita")
plt.show()

In [ ]:
print("=== Fin del análisis ===")
print(f"\nResultados guardados en: {config.output_dir.resolve()}")